<div align="center">

# NNDL Final Project: Flight Delay Forecasting

**Kevin Brugnera · Sara Pasquato · Libero Pollini**

IDs: 2196578 · (inserite) · 2206131

</div>

---

First and foremost, we explore the 2022 chain dataset.

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
from google.colab import files # opens interactive window to choose file(s) to upload files
import os

uploaded = files.upload()

for name in uploaded.keys():
    print("Uploaded:", name)

In [ ]:
year = 2022
split_types = ['train', 'val', 'test']

loaded_data = {}

for split in split_types:
    file_path = f'data/chain/{year}/{split}_flight_chain_{year}.pt'
    loaded_data[split] = torch.load(file_path, weights_only=False)
    
    print(f"--- File: (split: {split}, year: {year}) ---")
    print("Data Type:", type(loaded_data[split]))

Samples are fewer than those reported in [Aeolus](https://arxiv.org/pdf/2510.26616) because they filtered to retain only coherent flight chains operated by the same aircraft (see Table 8, page 17).

In [ ]:
N_train = len(loaded_data["train"])
N_val = len(loaded_data["val"])
N_test = len(loaded_data["test"])

N_chains = N_train + N_val + N_test

print(f"Total number of flight chains: {N_chains}")
print(f"Train samples: {N_train} ({N_train*100/N_chains:.2f}%)")
print(f"Validation samples: {N_val} ({N_val*100/N_chains:.2f}%)")
print(f"Test samples: {N_test} ({N_test*100/N_chains:.2f}%)")


Example of a flight chain sample

In [ ]:
# 1. Extract the first sample from the training dataset
sample = loaded_data["train"][0]

print(f"Sample Type: {type(sample)}")
print(f"Total components in the sample tuple: {len(sample)}\n")
print("-" * 60)

# 2. Unpack each component of the 5-element tuple into properly named variables
dense_feat, sparse_feat, labels, valid_lens, delays = sample

# 3. Print details and relative values for each component with descriptive labels based on source code
print("1. Dense Features (Continuous / Meteorological Features):")
print(f"   - Shape: {dense_feat.shape} (Sequence Length x 7)")
print(f"   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD', 'FLIGHTS']")
print(f"    - Values:\n{dense_feat}\n")

print("2. Sparse Features (Categorical / Temporal Features):")
print(f"   - Shape: {sparse_feat.shape} (Sequence Length x 8)")
print(f"   - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']")
print(f"    - Values:\n{sparse_feat}\n")

print("3. Binary Labels (Flight Delay Indicators > 15 mins):")
print(f"   - Shape: {labels.shape} (Sequence Length x 2)")
print(f"   - [(ARR_DELAY > 15), (DEP_DELAY > 15)]")
print(f"   - Values:\n{labels}\n")

print("4. Valid Sequence Lengths (Metadata):")
print(f"   - Description: Effective number of valid flights in the chain before padding")
print(f"   - Shape: {valid_lens.shape}")
print(f"   - Values: {valid_lens}\n")

print("5. Raw Delays (Ground Truth):")
print(f"   - Shape: {delays.shape} (Sequence Length x 2)")
print(f"   - Cols: ['ARR_DELAY', 'DEP_DELAY']")
print(f"   - Values:\n{delays}")
print("-" * 60)

Fist LSTM model on fixed length chains (6 flights)

In [ ]:
# Filtering

# 1. Define the exact target length required
target_length = 5

# 2. Initialize a dictionary to store the filtered datasets
filtered_loaded_data = {}

# 3. Iterate through each split ('train', 'val', 'test') in the loaded dataset
for split_name, dataset in loaded_data.items():
    print(f"Processing split: {split_name} (Original samples: {len(dataset)})")
    
    # Filter samples where the sequence length (dense_feat shape[0]) equals target_length
    filtered_samples = [
        sample for sample in dataset 
        if sample[0].shape[0] == target_length
    ]
    
    print(f"-> Filtered samples (length == {target_length}): {len(filtered_samples)}")
    
    # If samples match the condition, reconstruct into a TensorDataset
    if len(filtered_samples) > 0:
        dense_list = torch.stack([s[0] for s in filtered_samples])
        sparse_list = torch.stack([s[1] for s in filtered_samples])
        labels_list = torch.stack([s[2] for s in filtered_samples])
        valid_lens_list = torch.stack([s[3] for s in filtered_samples])
        delays_list = torch.stack([s[4] for s in filtered_samples])
        
        filtered_loaded_data[split_name] = TensorDataset(
            dense_list, sparse_list, labels_list, valid_lens_list, delays_list
        )
    else:
        # Keep an empty or None placeholder if no samples match
        filtered_loaded_data[split_name] = None

print("\nFiltering complete. All splits are stored in 'filtered_loaded_data'.")

LSTM with 6 recurrent unit + linear regression layer

In [ ]:
class FlightChainLSTM(nn.Module):
    def __init__(self, dense_input_dim, sparse_input_dim, hidden_dim=6, output_dim=2):
        """
        Params:
        - dense_input_dim: Number of continuous features (e.g., 7 meteorological/dense features).
        - sparse_input_dim: Number of categorical/sparse features (e.g., 8 discrete attributes).
        - hidden_dim: Number of hidden units in the LSTM layer (set to 6 as requested).
        - output_dim: Number of regression outputs (e.g., 2 for arrival and departure delays).
        """
        super(FlightChainLSTM, self).__init__()
        
        # Total input dimension combining dense and sparse features per time step
        self.total_input_dim = dense_input_dim + sparse_input_dim
        
        self.hidden_dim = hidden_dim
        
        # 1. LSTM Layer configured with 6 hidden units and batch_first=True
        # Input shape expected: [batch_size, sequence_length (6), total_input_dim]
        self.lstm = nn.LSTM(
            input_size=self.total_input_dim,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )
        
        # 2. Fully Connected Regression Layer to map LSTM outputs to the target delay values
        self.regressor = nn.Linear(self.hidden_dim, output_dim)
        
    def forward(self, dense_feat, sparse_feat):
        """
        Params:
        - dense_feat: Tensor of shape [batch_size, 6, 7]
        - sparse_feat: Tensor of shape [batch_size, 6, 8] (converted to float for concatenation)
        """
        x = torch.cat((dense_feat, sparse_feat.float()), dim=2)
        lstm_out, (hn, cn) = self.lstm(x)re
        predictions = self.regressor(lstm_out)
        
        return predictions

In [ ]:
# --- Example ---

DENSE_DIM = 7  # O_TEMP, D_TEMP, O_PRCP, D_PRCP, O_WSPD, D_WSPD, FLIGHTS
SPARSE_DIM = 8 # MONTH, DAY_OF_WEEK, CRS_ARR_TIME_HOUR, etc.
HIDDEN_UNITS = 6 # Set strictly to 6 units
OUTPUT_REGRESSION_DIM = 2 # ARR_DELAY and DEP_DELAY

# Instantiate the model
model = FlightChainLSTM(
    dense_input_dim=DENSE_DIM,
    sparse_input_dim=SPARSE_DIM,
    hidden_dim=HIDDEN_UNITS,
    output_dim=OUTPUT_REGRESSION_DIM
)

print(model)

In [ ]:
# 1. Hyperparameters & Configuration
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# 2. Create DataLoaders from your filtered splits
train_loader = DataLoader(filtered_loaded_data['train'], batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(filtered_loaded_data['val'], batch_size=BATCH_SIZE, shuffle=False)

# 3. Instantiate the Model, Loss, and Optimizer
model = FlightChainLSTM(
    dense_input_dim=7,
    sparse_input_dim=8,
    hidden_dim=6,
    output_dim=2
).to(DEVICE)

# MSE Loss is standard for regression tasks (predicting continuous delay values)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    running_train_loss = 0.0
    
    for batch in train_loader:
        dense_feat, sparse_feat, labels, valid_lens, targets = [tensor.to(DEVICE) for tensor in batch]
        
        # Zero the gradients from the previous step
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(dense_feat, sparse_feat)
        
        # oloss
        loss = criterion(predictions, targets.float())
        
        # Backpropagation
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        running_train_loss += loss.item() * dense_feat.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    
    # 5. Validation Loop
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            dense_feat, sparse_feat, labels, valid_lens, targets = [tensor.to(DEVICE) for tensor in batch]
            
            predictions = model(dense_feat, sparse_feat)
            loss = criterion(predictions, targets.float())
            
            running_val_loss += loss.item() * dense_feat.size(0)
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

print("Training and evaluation process completed successfully.")